# 6.1 · Graphs: representation and traversal

*Graphs* · 6 exercises

Store a graph as adjacency lists or a matrix, and visit it depth-first or breadth-first.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-6.1) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-6.1) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-6.1)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
from collections import deque

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 6.1.1 · Adjacency-list graph with a hash map  `Core`

**Task.** Write `add_edge(graph, a, b, directed=False)` for a graph stored as `{vertex: [neighbours]}`. Add missing vertices automatically and avoid duplicate edges.

**Example.** `g = {}; add_edge(g, 'Ames', 'Bern')` -> `{'Ames': ['Bern'], 'Bern': ['Ames']}`

**Big O.** O(D) for the duplicate check, where D is the vertex's degree. A set of neighbours would make it O(1).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-1)

In [ ]:
# 6.1.1 Adjacency-list graph with a hash map
def add_edge(graph, a, b, directed=False):
    raise NotImplementedError

### 6.1.2 · Vertex objects with adjacency lists  `Core`

**Task.** Create `Vertex(value)` with an `adjacent` list and `add_adjacent(vertex)` that links both ways (friendship is mutual) without recursing forever.

**Example.** `ames.add_adjacent(bern)` -> bern is in ames.adjacent and ames is in bern.adjacent

**Big O.** O(D) per link because of the membership check.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-2)

In [ ]:
# 6.1.2 Vertex objects with adjacency lists
class Vertex:
    def __init__(self, value):
        raise NotImplementedError

    def add_adjacent(self, vertex):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
def make_towns():
    ames, bern, cork, dover = (Vertex(n) for n in ('Ames', 'Bern', 'Cork', 'Dover'))
    ames.add_adjacent(bern)
    ames.add_adjacent(cork)
    bern.add_adjacent(dover)
    return ames, bern, cork, dover

In [ ]:
def test_ex_6_1_1():
    g = {}
    add_edge(g, 'Ames', 'Bern')
    assert g == {'Ames': ['Bern'], 'Bern': ['Ames']}
    add_edge(g, 'Ames', 'Bern')
    add_edge(g, 'Bern', 'Cork', directed=True)
    assert g == {'Ames': ['Bern'], 'Bern': ['Ames', 'Cork'], 'Cork': []}


test_ex_6_1_1()
print('6.1.1 passed')

In [ ]:
def test_ex_6_1_2():
    ames, bern, cork, dover = make_towns()
    assert ames in bern.adjacent
    assert bern in ames.adjacent
    assert dover in bern.adjacent
    ames.add_adjacent(bern)
    assert ames.adjacent.count(bern) == 1


test_ex_6_1_2()
print('6.1.2 passed')

### 6.1.3 · Depth-first traversal  `Core`

**Task.** Write `dfs_traverse(vertex, visited=None)` returning vertex values in visit order. Mark each vertex visited before recursing into its neighbours.

**Example.** Ames-Bern, Ames-Cork, Bern-Dover -> `['Ames', 'Bern', 'Dover', 'Cork']`

**Big O.** O(V + E). Without the visited set, a cycle would loop forever.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-3)

In [ ]:
# 6.1.3 Depth-first traversal
def dfs_traverse(vertex, visited=None):
    raise NotImplementedError

In [ ]:
def test_ex_6_1_3():
    ames, *_ = make_towns()
    assert dfs_traverse(ames) == ['Ames', 'Bern', 'Dover', 'Cork']


test_ex_6_1_3()
print('6.1.3 passed')

### 6.1.4 · Depth-first search  `Core`

**Task.** Write `dfs_search(vertex, target_value, visited=None)` returning the matching `Vertex` or `None`.

**Example.** From Ames, search 'Dover' -> the Dover vertex

**Big O.** O(V + E) worst case.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-4)

In [ ]:
# 6.1.4 Depth-first search
def dfs_search(vertex, target_value, visited=None):
    raise NotImplementedError

In [ ]:
def test_ex_6_1_4():
    ames, bern, cork, dover = make_towns()
    assert dfs_search(ames, 'Dover') is dover
    assert dfs_search(ames, 'Zed') is None


test_ex_6_1_4()
print('6.1.4 passed')

### 6.1.5 · Breadth-first traversal  `Core`

**Task.** Write `bfs_traverse(start)` using a queue: mark the start visited, then repeatedly dequeue a vertex and enqueue its unvisited neighbours.

**Example.** Same graph as 6.1.3 -> `['Ames', 'Bern', 'Cork', 'Dover']`

**Big O.** O(V + E).

**Note.** BFS spreads out in rings, so it finds close vertices first; DFS dives deep first.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-5)

In [ ]:
# 6.1.5 Breadth-first traversal
def bfs_traverse(start):
    raise NotImplementedError

In [ ]:
def test_ex_6_1_5():
    ames, *_ = make_towns()
    assert bfs_traverse(ames) == ['Ames', 'Bern', 'Cork', 'Dover']


test_ex_6_1_5()
print('6.1.5 passed')

### 6.1.6 · Count connected groups  `Practice`

**Task.** Write `count_components(graph)` returning how many separate groups an undirected dict graph has.

**Example.** `{'a': ['b'], 'b': ['a'], 'c': [], 'd': ['e'], 'e': ['d']}` -> `3`

**Big O.** O(V + E): each vertex is explored once across all searches.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-1-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-1-6)

In [ ]:
# 6.1.6 Count connected groups
def count_components(graph):
    raise NotImplementedError

In [ ]:
def test_ex_6_1_6():
    assert count_components({'a': ['b'], 'b': ['a'], 'c': [], 'd': ['e'], 'e': ['d']}) == 3


test_ex_6_1_6()
print('6.1.6 passed')